# Transformer Attention Regressor — kGain Prediction
Focal regression loss · seed=19 · batch=8192 · epochs=500

In [1]:
import os
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive — prevents plt.show() blocking
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2026-05-01 09:05:42.259600: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-01 09:05:42.259639: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-01 09:05:42.260683: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-01 09:05:42.265482: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-01 09:05:42.887976: W tensorflow/compiler/tf2

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=40000)]
        )
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), 'Physical GPUs,', len(logical_gpus), 'Logical GPUs')
    except RuntimeError as e:
        print(e)

1 Physical GPUs, 1 Logical GPUs


2026-05-01 09:05:44.357788: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 40000 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:81:00.0, compute capability: 8.6


In [3]:
def set_seed(seed=42, enable_tf_determinism=True):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['CUDA_VISIBLE_DEVICES'] = ''
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    if enable_tf_determinism:
        try:
            tf.config.experimental.enable_op_determinism(True)
        except Exception:
            pass

my_seed = 19
set_seed(my_seed)

## Load data
Update `DATA_DIR` and `LAB_CSV` paths for your server.

In [4]:
# ── UPDATE THESE PATHS ──────────────────────────────────────────────────────
DATA_DIR = '../../../data'   # path to kgain_all_population_wt.csv
LAB_CSV  = '../../../data/Lab_ltte_with_af1.csv'
# ────────────────────────────────────────────────────────────────────────────

file_path = os.path.join(DATA_DIR, 'kgain_all_population_wt.csv')
df = pd.read_csv(file_path)[['ref_flank_seq', 'ALT', 'wild_type_kGain']]
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Loaded {len(df)} rows')
df.head()

Loaded 34391 rows


,ref_flank_seq,ALT,wild_type_kGain
0,GTCCACCCGCCGTATTGCG,T,-8.764470
1,ACGGAACGGCTGGCCATTA,T,-11.203187
2,CCAATATCAACATTGTCGC,G,-2.844573
3,CCGGCTCCGGTCGCCAATG,A,-9.244451
4,GCAGCGTATAGCGCGTGGT,G,7.647503


## Encoding

In [5]:
def one_hot_encode(seq, base_to_int={'A': 0, 'C': 1, 'G': 2, 'T': 3}):
    encoded = np.zeros((len(seq), 4))
    for i, base in enumerate(seq):
        if base in base_to_int:
            encoded[i, base_to_int[base]] = 1
    return encoded


def encode_ref_minus_alt(ref_seq, alt_allele, mut_idx=9):
    arr = []
    for i, base in enumerate(ref_seq):
        if i == mut_idx:
            ohe = np.array(one_hot_encode(base) - one_hot_encode(alt_allele))
        else:
            ohe = np.array(one_hot_encode(base))
        arr.append(ohe)
    return np.array(arr)


def encode_row(row):
    ref_seq   = row['ref_flank_seq']
    alt_allele = row['ALT']
    mut_idx   = len(ref_seq) // 2
    return encode_ref_minus_alt(ref_seq, alt_allele, mut_idx)

## Prepare inputs

In [6]:
X_encoded = np.stack(df.apply(encode_row, axis=1)).reshape(-1, 19, 4)
y = df['wild_type_kGain'].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)
print(f'X_encoded: {X_encoded.shape}  |  X_train: {X_train.shape}  |  X_test: {X_test.shape}')

X_encoded: (34391, 19, 4)  |  X_train: (27512, 19, 4)  |  X_test: (6879, 19, 4)


## Loss function

In [7]:
def focal_regression_loss(gamma=2.0):
    def loss(y_true, y_pred):
        y_pred = tf.cast(y_pred, y_true.dtype)
        err = tf.abs(y_true - y_pred)
        err = tf.maximum(err, keras.backend.epsilon())
        fl  = tf.pow(err, gamma) * err
        return tf.reduce_mean(fl)
    loss.__name__ = f'focal_regression_loss_g{gamma}'
    return loss

## Model architecture

In [8]:
ATTN_SEQ_LEN      = 19
ATTN_ONEHOT_DIM   = 4
ATTN_EMB_DIM      = 32
ATTN_NUM_HEADS    = 4
ATTN_FF_DIM       = 32
ATTN_NUM_LAYERS   = 2
ATTN_DROPOUT_RATE = 0.1
gamma             = 2


def build_attention_regressor(
    seq_len=ATTN_SEQ_LEN, onehot_dim=ATTN_ONEHOT_DIM, emb_dim=ATTN_EMB_DIM,
    num_heads=ATTN_NUM_HEADS, ff_dim=ATTN_FF_DIM, num_layers=ATTN_NUM_LAYERS,
    dropout_rate=ATTN_DROPOUT_RATE, return_attention=False, seed=None
):
    inp = keras.Input(shape=(seq_len, onehot_dim), name='seq_onehot_input')
    x = layers.Dense(
        emb_dim,
        kernel_initializer=keras.initializers.GlorotUniform(seed=seed),
        bias_initializer=keras.initializers.Zeros()
    )(inp)
    pos_idx = tf.range(seq_len)[tf.newaxis, :]
    pos_emb = layers.Embedding(
        input_dim=seq_len, output_dim=emb_dim,
        embeddings_initializer=keras.initializers.GlorotUniform(seed=seed)
    )(pos_idx)
    pos_emb = tf.squeeze(pos_emb, axis=0)
    x = x + pos_emb
    x = layers.Dropout(dropout_rate, seed=seed)(x)

    attention_scores_outputs = []
    for i in range(num_layers):
        attn_layer = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=emb_dim, name=f'my_attention_{i}',
            kernel_initializer=keras.initializers.GlorotUniform(seed=seed),
            bias_initializer=keras.initializers.Zeros()
        )
        attn_out, attn_scores = attn_layer(x, x, return_attention_scores=True)
        attention_scores_outputs.append(attn_scores)
        attn_out = layers.Dropout(dropout_rate, seed=seed)(attn_out)
        x = layers.Add()([x, attn_out])
        x = layers.LayerNormalization()(x)
        ff = layers.Dense(
            ff_dim, activation='relu',
            kernel_initializer=keras.initializers.GlorotUniform(seed=seed),
            bias_initializer=keras.initializers.Zeros()
        )(x)
        ff = layers.Dropout(dropout_rate, seed=seed)(ff)
        ff = layers.Dense(
            emb_dim,
            kernel_initializer=keras.initializers.GlorotUniform(seed=seed),
            bias_initializer=keras.initializers.Zeros()
        )(ff)
        x = layers.Add()([x, ff])
        x = layers.LayerNormalization()(x)

    x   = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(
        1, activation='linear', name='regression_output',
        kernel_initializer=keras.initializers.GlorotUniform(seed=seed),
        bias_initializer=keras.initializers.Zeros()
    )(x)

    if return_attention:
        return keras.Model(inp, [out] + attention_scores_outputs)
    return keras.Model(inp, out)

## Train

In [9]:
model = build_attention_regressor(return_attention=False, seed=my_seed)
model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss=focal_regression_loss(gamma=gamma),
    metrics=['mae', 'mse']
)
model.summary()

2026-05-01 09:05:45.723875: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 39.06GiB (41943040000 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2026-05-01 09:05:45.725124: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 35.16GiB (37748736000 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2026-05-01 09:05:45.726349: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 31.64GiB (33973862400 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2026-05-01 09:05:45.727567: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 28.48GiB (30576476160 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2026-05-01 09:05:45.728780: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 25.63GiB (27518828544 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2026-05-01 09:05:45.

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ seq_onehot_input    │ (None, 19, 4)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 19, 32)    │        160 │ seq_onehot_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 19, 32)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 19, 32)    │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ my_attention_0      │ [(None, 19, 32),  │     16,800 │ dropout[0][0],    │
│ (MultiHeadAttentio… │ (None, 4, 19,     │            │ dropout[0][0]     │
│                     │ 19)]              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 19, 32)    │          0 │ my_attention_0[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 19, 32)    │          0 │ dropout[0][0],    │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 19, 32)    │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 19, 32)    │      1,056 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 19, 32)    │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 19, 32)    │      1,056 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 19, 32)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 19, 32)    │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ my_attention_1      │ [(None, 19, 32),  │     16,800 │ layer_normalizat… │
│ (MultiHeadAttentio… │ (None, 4, 19,     │            │ layer_normalizat… │
│                     │ 19)]              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 19, 32)    │          0 │ my_attention_1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 19, 32)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 19, 32)    │         64 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 19, 32)    │      1,056 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 19, 32)    │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 38,273 (149.50 KB)

 Trainable params: 38,273 (149.50 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(
    X_encoded, y,
    epochs=500,
    verbose=1,
    batch_size=8192,
    validation_split=0.2
)

Epoch 1/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 148ms/step - loss: 310.7107 - mae: 4.6138 - mse: 33.3750 - val_loss: 215.5484 - val_mae: 4.0253 - val_mse: 25.6336
Epoch 2/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 241.7749 - mae: 4.2221 - mse: 28.0715 - val_loss: 220.0820 - val_mae: 4.0823 - val_mse: 26.2573
Epoch 3/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 239.2972 - mae: 4.2045 - mse: 27.8674 - val_loss: 215.5137 - val_mae: 4.0409 - val_mse: 25.7659
Epoch 4/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 238.9419 - mae: 4.2033 - mse: 27.8429 - val_loss: 220.6696 - val_mae: 4.0876 - val_mse: 26.3204
Epoch 5/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 239.0248 - mae: 4.2036 - mse: 27.8666 - val_loss: 214.8776 - val_mae: 4.0372 - val_mse: 25.7204
Epoch 6/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 238.3852 - mae: 4.2001 - mse: 27.7946 - val_loss: 215.7902 - val_mae: 4.0495 - val_mse: 25.8582
Epoch 7/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 237.1658 - mae: 4

## Save weights & history

In [11]:
model.save_weights('attn_regressor_focal.weights.h5')
print('Weights saved: attn_regressor_focal.weights.h5')

with open('history.pkl', 'wb') as f:
    pickle.dump(history.history, f)
print('History saved: history.pkl')

Weights saved: attn_regressor_focal.weights.h5
History saved: history.pkl


## Loss curve

In [12]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history.history['loss'],     label='Train loss')
ax.plot(history.history['val_loss'], label='Val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Focal loss')
ax.legend(); ax.set_title('Training curve')
plt.tight_layout()
plt.savefig('Transformer_loss_curve_custom.pdf', dpi=300)
plt.close()
print('Saved: Transformer_loss_curve_custom.pdf')

Saved: Transformer_loss_curve_custom.pdf


## LTEE evaluation (held-out 20%)

In [13]:
y_pred_ltee = model.predict(X_test).ravel()

mse_ltee  = mean_squared_error(y_test, y_pred_ltee)
mae_ltee  = mean_absolute_error(y_test, y_pred_ltee)
r2_ltee   = r2_score(y_test, y_pred_ltee)
corr_ltee, _ = pearsonr(y_test, y_pred_ltee)

print(f'LTEE  corr={corr_ltee:.4f}  R2={r2_ltee:.4f}  MAE={mae_ltee:.4f}  MSE={mse_ltee:.4f}')

215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
LTEE  corr=0.8150  R2=0.6414  MAE=2.4853  MSE=9.9171


## Lab evaluation

In [15]:
lab_df = pd.read_csv(LAB_CSV)
lab_df = lab_df[['ref_flank', 'REF', 'POS', 'ALT', 'kgain_wt', 'GENE']].dropna().drop_duplicates()
lab_df.reset_index(drop=True, inplace=True)
lab_df.columns = ['ref_flank_seq', 'REF', 'POS', 'ALT', 'kgain_wt', 'GENE']
lab_df['ref_flank_seq'] = lab_df['ref_flank_seq'].apply(lambda x: x[1:-1])

X_encoded_lab = np.stack(lab_df.apply(encode_row, axis=1)).reshape(-1, 19, 4)
y_lab = lab_df['kgain_wt'].values.astype(np.float32)

y_pred_lab = model.predict(X_encoded_lab).ravel()
lab_df['predicted_kgain_wt'] = y_pred_lab

mse_lab  = mean_squared_error(y_lab, y_pred_lab)
mae_lab  = mean_absolute_error(y_lab, y_pred_lab)
r2_lab   = r2_score(y_lab, y_pred_lab)
corr_lab, _ = pearsonr(y_lab, y_pred_lab)

print(f'Lab   corr={corr_lab:.4f}  R2={r2_lab:.4f}  MAE={mae_lab:.4f}  MSE={mse_lab:.4f}')
print(f'\ncorr_lab, corr_ltee = ({corr_lab:.10f}, {corr_ltee:.10f})')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Lab   corr=0.8209  R2=0.6709  MAE=2.4797  MSE=11.5054

corr_lab, corr_ltee = (0.8209180949, 0.8149565871)
